In [1]:
%%writefile recognizeidentifiers.cpp

#include <stdio.h>
#include <ctype.h>
#include <string.h>

#define MAX 100

// Check if a string is a valid identifier
int isValidIdentifier(char *str) {
    if (str[0] == '\0')
        return 0;

    if (!isalpha(str[0]) && str[0] != '_')
        return 0;

    for (int i = 1; str[i]; i++) {
        if (!isalnum(str[i]) && str[i] != '_')
            return 0;
    }
    return 1;
}

// Check if a word is a C keyword
int isKeyword(char *word) {
    const char *keywords[] = {
        "auto", "break", "case", "char", "const", "continue", "default", "do",
        "double", "else", "enum", "extern", "float", "for", "goto", "if",
        "inline", "int", "long", "register", "restrict", "return", "short", "signed",
        "sizeof", "static", "struct", "switch", "typedef", "union", "unsigned", "void",
        "volatile", "while"
    };

    int numKeywords = sizeof(keywords) / sizeof(keywords[0]);

    for (int i = 0; i < numKeywords; i++) {
        if (strcmp(word, keywords[i]) == 0)
            return 1;
    }
    return 0;
}

// Extract identifiers from file
void extractIdentifiers(char *filename) {
    FILE *fp = fopen(filename, "r");

    if (!fp) {
        printf("Error opening file\n");
        return;
    }

    char token[MAX];
    int j = 0;
    char c;

    printf("Checking valid Identifiers in File %s:\n\n", filename);

    while ((c = fgetc(fp)) != EOF) {
        if (isalpha(c) || c == '_') {
            token[j++] = c;
        } else {
            if (j > 0) {
                token[j] = '\0';

                if (isValidIdentifier(token) && !isKeyword(token)) {
                    printf("%s\n", token);
                }
                j = 0;
            }
        }
    }

    // Handle last token
    if (j > 0) {
        token[j] = '\0';
        if (isValidIdentifier(token) && !isKeyword(token)) {
            printf("%s\n", token);
        }
    }

    fclose(fp);
}

// Main function
int main() {
    extractIdentifiers("Check.c");
    return 0;
}

SyntaxError: invalid syntax (1393176445.py, line 7)

In [2]:
%%writefile countlineswords.cpp

#include <stdio.h>
#include <ctype.h>

// Function to count lines, words, and characters
void countStats(char *filename) {
    FILE *fp = fopen(filename, "r");

    if (!fp) {
        printf("Error opening file\n");
        return;
    }

    int lines = 0, words = 0, characters = 0;
    char c;
    int inWord = 0;

    while ((c = fgetc(fp)) != EOF) {
        characters++;

        if (c == '\n') {
            lines++;
            inWord = 0;
        }
        else if (isspace(c)) {
            if (inWord) {
                words++;
                inWord = 0;
            }
        }
        else {
            inWord = 1;
        }
    }

    // Count last word if file doesn't end with space
    if (inWord)
        words++;

    // Count last line if file is not empty
    if (characters > 0)
        lines++;

    fclose(fp);

    printf("Checking in file %s:\n\n", filename);
    printf("Lines: %d\n", lines);
    printf("Words: %d\n", words);
    printf("Characters: %d\n", characters);
}

// Main function
int main() {
    countStats("Check.c");
    return 0;
}

SyntaxError: unterminated string literal (detected at line 35) (112989235.py, line 35)

In [3]:
%%writefile lineiscomment.cpp

#include <stdio.h>
#include <stdlib.h>
#include <ctype.h>

#define MAX 1024

// Check if a line is a single-line comment
int isCommentLine(char *line) {
    while (*line && isspace(*line))
        line++;

    return (line[0] == '/' && line[1] == '/');
}

// Function to remove comments
void removeComments(char *input, char *output) {
    FILE *in = fopen(input, "r");
    FILE *out = fopen(output, "w");

    if (!in || !out) {
        printf("Error opening files\n");
        return;
    }

    char line[MAX];
    int inComment = 0;

    while (fgets(line, sizeof(line), in)) {

        // Handle multi-line comments already started
        if (inComment) {
            for (int i = 0; line[i]; i++) {
                if (line[i] == '*' && line[i + 1] == '/') {
                    inComment = 0;
                    i += 2;

                    while (line[i]) {
                        if (line[i] != '\n')
                            fputc(line[i], out);
                        i++;
                    }

                    if (line[i - 1] == '\n')
                        fputc('\n', out);

                    break;
                }
            }
            continue;
        }

        char result[MAX] = "";
        int j = 0;

        for (int i = 0; line[i]; i++) {

            // Start of multi-line comment
            if (line[i] == '/' && line[i + 1] == '*') {
                inComment = 1;
                i += 2;

                while (line[i]) {
                    if (line[i] == '*' && line[i + 1] == '/') {
                        inComment = 0;
                        i += 2;
                        break;
                    }
                    i++;
                }
                i--; // adjust index
            }

            // Single-line comment
            else if (line[i] == '/' && line[i + 1] == '/') {
                break;
            }

            // Normal character
            else if (!inComment) {
                result[j++] = line[i];
            }
        }

        result[j] = '\0';

        // Trim trailing spaces
        while (j > 0 && isspace(result[j - 1])) {
            result[--j] = '\0';
        }

        if (j > 0)
            fprintf(out, "%s\n", result);
    }

    fclose(in);
    fclose(out);
}

// Main function
int main() {
    removeComments("Check.c", "sample_no_comments.c");
    printf("Comments removed successfully\n");
    return 0;
}

Writing lineiscomment.cpp


In [3]:
%%writefile identifyall.cpp

#include <stdio.h>
#include <ctype.h>
#include <string.h>

#define MAX_TOKEN 100

// List of C keywords
const char *keywords[] = {
    "auto", "break", "case", "char", "const", "continue", "default", "do",
    "double", "else", "enum", "extern", "float", "for", "goto", "if",
    "inline", "int", "long", "register", "restrict", "return", "short", "signed",
    "sizeof", "static", "struct", "switch", "typedef", "union", "unsigned", "void",
    "volatile", "while"
};

// Check if word is keyword
int isKeyword(char *word) {
    int count = sizeof(keywords) / sizeof(keywords[0]);

    for (int i = 0; i < count; i++) {
        if (strcmp(word, keywords[i]) == 0)
            return 1;
    }
    return 0;
}

// Check if character is operator
int isOperator(char c) {
    char op[] = "+-*/%=<>!&|^~?:";

    for (int i = 0; op[i]; i++) {
        if (c == op[i])
            return 1;
    }
    return 0;
}

// Check if character is punctuation
int isPunct(char c) {
    char p[] = "()[]{},.;";

    for (int i = 0; p[i]; i++) {
        if (c == p[i])
            return 1;
    }
    return 0;
}

// Check identifier
int isIdentifier(char *str) {
    if (!isalpha(str[0]) && str[0] != '_')
        return 0;

    for (int i = 1; str[i]; i++) {
        if (!isalnum(str[i]) && str[i] != '_')
            return 0;
    }
    return 1;
}

// Check constant (numbers)
int isConstant(char *str) {
    if (str[0] == '\0')
        return 0;

    for (int i = 0; str[i]; i++) {
        if (!isdigit(str[i]) && str[i] != '.')
            return 0;
    }
    return 1;
}

// Analyze file
void analyze(FILE *fp) {
    char token[MAX_TOKEN];
    int j = 0;
    char c;

    while ((c = fgetc(fp)) != EOF) {

        // Space → process token
        if (isspace(c)) {
            if (j > 0) {
                token[j] = '\0';

                if (isKeyword(token))
                    printf("KEYWORD: %s\n", token);
                else if (isConstant(token))
                    printf("CONSTANT: %s\n", token);
                else if (isIdentifier(token))
                    printf("IDENTIFIER: %s\n", token);

                j = 0;
            }
        }

        // Operator
        else if (isOperator(c)) {
            if (j > 0) {
                token[j] = '\0';

                if (isKeyword(token))
                    printf("KEYWORD: %s\n", token);
                else if (isConstant(token))
                    printf("CONSTANT: %s\n", token);
                else if (isIdentifier(token))
                    printf("IDENTIFIER: %s\n", token);

                j = 0;
            }
            printf("OPERATOR: %c\n", c);
        }

        // Punctuation
        else if (isPunct(c)) {
            if (j > 0) {
                token[j] = '\0';

                if (isKeyword(token))
                    printf("KEYWORD: %s\n", token);
                else if (isConstant(token))
                    printf("CONSTANT: %s\n", token);
                else if (isIdentifier(token))
                    printf("IDENTIFIER: %s\n", token);

                j = 0;
            }
            printf("PUNCTUATION: %c\n", c);
        }

        // Build token
        else {
            if (j < MAX_TOKEN - 1)
                token[j++] = c;
        }
    }

    // Handle last token
    if (j > 0) {
        token[j] = '\0';

        if (isKeyword(token))
            printf("KEYWORD: %s\n", token);
        else if (isConstant(token))
            printf("CONSTANT: %s\n", token);
        else if (isIdentifier(token))
            printf("IDENTIFIER: %s\n", token);
    }
}

// Main function
int main() {
    FILE *fp = fopen("Check.c", "r");

    if (!fp) {
        printf("Error opening file\n");
        return 1;
    }

    analyze(fp);
    fclose(fp);

    return 0;
}

In [4]:
%%writefile cat count.l
lex code

%{
#include <stdio.h>

int lines = 0;
int words = 0;
int spaces = 0;
int characters = 0;
%}

/* Definitions */
%%
\n              { lines++; characters++; }
[ \t]+          { spaces += yyleng; characters += yyleng; }
[a-zA-Z0-9]+    { words++; characters += yyleng; }
.               { characters++; }
%%

// Main function
int main(int argc, char *argv[]) {
    if (argc > 1) {
        yyin = fopen(argv[1], "r");
        if (!yyin) {
            printf("Error opening file\n");
            return 1;
        }
    }

    yylex();

    printf("Lines: %d\n", lines);
    printf("Words: %d\n", words);
    printf("Spaces: %d\n", spaces);
    printf("Characters: %d\n", characters);

    return 0;
}

// Required by Lex
int yywrap() {
    return 1;
}


flex count.l      → generates lex.yy.c
gcc lex.yy.c -lfl → compile C file

keyboard i/p
./output

i/p from a file
./output filename.txt


UsageError: unrecognized arguments: count.l


In [ ]:
%%writefile cat count2.l
lex code


%{
#include <stdio.h>

int if_count = 0;
int else_count = 0;
int while_count = 0;

int plus_count = 0;
int minus_count = 0;
int mul_count = 0;
int div_count = 0;
%}

/* Rules section */
%%
"if"        { if_count++; }
"else"      { else_count++; }
"while"     { while_count++; }

"+"         { plus_count++; }
"-"         { minus_count++; }
"*"         { mul_count++; }
"/"         { div_count++; }

[ \t\n]+    { /* ignore whitespace */ }
.           { /* ignore other characters */ }
%%

// Main function
int main(int argc, char *argv[]) {

    if (argc > 1) {
        yyin = fopen(argv[1], "r");

        if (!yyin) {
            printf("Error opening file\n");
            return 1;
        }
    }

    yylex();

    printf("Keyword Counts:\n");
    printf("if: %d\n", if_count);
    printf("else: %d\n", else_count);
    printf("while: %d\n", while_count);

    printf("\nOperator Counts:\n");
    printf("+ : %d\n", plus_count);
    printf("- : %d\n", minus_count);
    printf("* : %d\n", mul_count);
    printf("/ : %d\n", div_count);

    return 0;
}

// Required by Lex
int yywrap() {
    return 1;
}


flex count.l      → generates lex.yy.c
gcc lex.yy.c -lfl → compile C file

keyboard i/p
./output

i/p from a file
./output filename.txt

In [ ]:
%%writefile constDFAfromRE.cpp

#include <bits/stdc++.h>
using namespace std;

/************* INFIX TO POSTFIX *************/
int prec(char c) {
    if (c == '*') return 3;
    if (c == '.') return 2;
    if (c == '|') return 1;
    return 0;
}

string addConcat(string re) {
    string r = "";
    for (int i = 0; i < re.size(); i++) {
        r += re[i];

        if (i + 1 < re.size()) {
            char c1 = re[i], c2 = re[i + 1];

            if ((c1 == 'a' || c1 == 'b' || c1 == '*' || c1 == ')') &&
                (c2 == 'a' || c2 == 'b' || c2 == '(')) {
                r += '.';
            }
        }
    }
    return r;
}

string infixToPostfix(string re) {
    re = addConcat(re);
    string post = "";
    stack<char> st;

    for (char c : re) {
        if (c == 'a' || c == 'b') {
            post += c;
        }
        else if (c == '(') {
            st.push(c);
        }
        else if (c == ')') {
            while (!st.empty() && st.top() != '(') {
                post += st.top();
                st.pop();
            }
            if (!st.empty()) st.pop(); // remove '('
        }
        else {
            while (!st.empty() && prec(st.top()) >= prec(c)) {
                post += st.top();
                st.pop();
            }
            st.push(c);
        }
    }

    while (!st.empty()) {
        post += st.top();
        st.pop();
    }

    return post;
}

/************* THOMPSON NFA *************/
struct State {
    map<char, vector<int>> trans;
    vector<int> eps;
};

vector<State> nfa;

pair<int, int> buildNFA(string post) {
    stack<pair<int, int>> st;

    for (char c : post) {
        if (c == 'a' || c == 'b') {
            int s = nfa.size();
            nfa.push_back(State());
            nfa.push_back(State());

            nfa[s].trans[c].push_back(s + 1);
            st.push({s, s + 1});
        }
        else if (c == '.') {
            auto n2 = st.top(); st.pop();
            auto n1 = st.top(); st.pop();

            nfa[n1.second].eps.push_back(n2.first);
            st.push({n1.first, n2.second});
        }
        else if (c == '|') {
            auto n2 = st.top(); st.pop();
            auto n1 = st.top(); st.pop();

            int s = nfa.size();
            nfa.push_back(State());
            int e = nfa.size();
            nfa.push_back(State());

            nfa[s].eps.push_back(n1.first);
            nfa[s].eps.push_back(n2.first);
            nfa[n1.second].eps.push_back(e);
            nfa[n2.second].eps.push_back(e);

            st.push({s, e});
        }
        else if (c == '*') {
            auto n = st.top(); st.pop();

            int s = nfa.size();
            nfa.push_back(State());
            int e = nfa.size();
            nfa.push_back(State());

            nfa[s].eps.push_back(n.first);
            nfa[s].eps.push_back(e);
            nfa[n.second].eps.push_back(n.first);
            nfa[n.second].eps.push_back(e);

            st.push({s, e});
        }
    }

    return st.top();
}

/************* NFA -> DFA *************/
set<int> epsClosure(int s) {
    stack<int> st;
    st.push(s);

    set<int> res;
    res.insert(s);

    while (!st.empty()) {
        int u = st.top();
        st.pop();

        for (int v : nfa[u].eps) {
            if (!res.count(v)) {
                res.insert(v);
                st.push(v);
            }
        }
    }

    return res;
}

set<int> moveSet(set<int> T, char c) {
    set<int> res;

    for (int u : T) {
        if (nfa[u].trans.count(c)) {
            for (int v : nfa[u].trans[c]) {
                auto cl = epsClosure(v);
                res.insert(cl.begin(), cl.end());
            }
        }
    }

    return res;
}

/************* MAIN *************/
int main() {
    string infix;
    cout << "Enter infix RE: ";
    cin >> infix;

    string postfix = infixToPostfix(infix);
    cout << "Postfix: " << postfix << endl;

    pair<int, int> p = buildNFA(postfix);

    set<int> start = epsClosure(p.first);

    map<set<int>, int> dfaID;
    vector<set<int>> dfaList;
    queue<set<int>> q;

    dfaID[start] = 0;
    dfaList.push_back(start);
    q.push(start);

    map<int, map<char, int>> DFATrans;

    while (!q.empty()) {
        auto T = q.front();
        q.pop();

        int id = dfaID[T];

        for (char c : {'a', 'b'}) {
            auto U = moveSet(T, c);

            if (!U.empty()) {
                if (!dfaID.count(U)) {
                    dfaID[U] = dfaList.size();
                    dfaList.push_back(U);
                    q.push(U);
                }
                DFATrans[id][c] = dfaID[U];
            }
        }
    }

    cout << "\nDFA Transition Table:\n";
    cout << "State |  a  |  b\n";
    cout << "-----------------\n";

    for (auto &row : DFATrans) {
        int s = row.first;

        cout << "  " << s << "   | ";
        cout << (row.second.count('a') ? to_string(row.second.at('a')) : "-") << "  | ";
        cout << (row.second.count('b') ? to_string(row.second.at('b')) : "-") << "\n";
    }

    cout << "\nStart state: 0\n";
    cout << "Accepting states:";

    for (auto &p2 : dfaID) {
        if (p2.first.count(p.second))
            cout << " " << p2.second;
    }

    cout << endl;

    return 0;
}

In [ ]:
%%writefile firstandfollow.cpp

#include <bits/stdc++.h>
using namespace std;

map<char, vector<string>> productions;
map<char, set<char>> FIRST, FOLLOW;
set<char> nonTerminals, terminals;

char startSymbol;

/* ---------- Utility ---------- */
bool isTerminal(char c) {
    return !(c >= 'A' && c <= 'Z') && c != '#';
}

/* ---------- FIRST ---------- */
void findFIRST(char nt) {
    for (string prod : productions[nt]) {

        // Case: epsilon
        if (prod == "#") {
            FIRST[nt].insert('#');
            continue;
        }

        for (int i = 0; i < prod.length(); i++) {
            char symbol = prod[i];

            if (isTerminal(symbol)) {
                FIRST[nt].insert(symbol);
                break;
            }
            else {
                findFIRST(symbol);

                for (char x : FIRST[symbol]) {
                    if (x != '#')
                        FIRST[nt].insert(x);
                }

                // If epsilon NOT present → stop
                if (FIRST[symbol].count('#') == 0)
                    break;

                // If last symbol and all had epsilon
                if (i == prod.length() - 1)
                    FIRST[nt].insert('#');
            }
        }
    }
}

/* ---------- FOLLOW ---------- */
void findFOLLOW(char nt) {

    // Start symbol gets $
    if (nt == startSymbol)
        FOLLOW[nt].insert('$');

    for (auto p : productions) {
        char lhs = p.first;

        for (string prod : p.second) {
            for (int i = 0; i < prod.length(); i++) {

                if (prod[i] == nt) {
                    bool epsilonFound = true;

                    // Check symbols after nt
                    for (int j = i + 1; j < prod.length(); j++) {
                        char next = prod[j];

                        if (isTerminal(next)) {
                            FOLLOW[nt].insert(next);
                            epsilonFound = false;
                            break;
                        }
                        else {
                            for (char x : FIRST[next]) {
                                if (x != '#')
                                    FOLLOW[nt].insert(x);
                            }

                            if (FIRST[next].count('#') == 0) {
                                epsilonFound = false;
                                break;
                            }
                        }
                    }

                    // If epsilon continues till end
                    if (epsilonFound && lhs != nt) {
                        findFOLLOW(lhs);

                        for (char x : FOLLOW[lhs])
                            FOLLOW[nt].insert(x);
                    }
                }
            }
        }
    }
}

/* ---------- MAIN ---------- */
int main() {
    int n;
    cout << "Enter number of productions: ";
    cin >> n;

    cout << "Enter productions (Example: E=TR):\n";

    for (int i = 0; i < n; i++) {
        string prod;
        cin >> prod;

        char lhs = prod[0];
        string rhs = prod.substr(2);

        productions[lhs].push_back(rhs);
        nonTerminals.insert(lhs);

        for (char c : rhs) {
            if (isTerminal(c))
                terminals.insert(c);
        }
    }

    // First production LHS = start symbol
    startSymbol = productions.begin()->first;

    // Compute FIRST
    for (char nt : nonTerminals)
        findFIRST(nt);

    // Compute FOLLOW
    for (char nt : nonTerminals)
        findFOLLOW(nt);

    /* ---------- OUTPUT ---------- */
    cout << "\nFIRST Sets:\n";
    for (char nt : nonTerminals) {
        cout << "FIRST(" << nt << ") = { ";
        for (char x : FIRST[nt])
            cout << x << " ";
        cout << "}\n";
    }

    cout << "\nFOLLOW Sets:\n";
    for (char nt : nonTerminals) {
        cout << "FOLLOW(" << nt << ") = { ";
        for (char x : FOLLOW[nt])
            cout << x << " ";
        cout << "}\n";
    }

    return 0;
}

In [ ]:
%%writefile operatorprecedence.cpp

#include <iostream>
#include <vector>
#include <map>
#include <set>
#include <stack>
#include <string>
using namespace std;

map<char, set<char>> FIRSTVT, LASTVT;
map<char, map<char, char>> precedence;
vector<string> productions;
set<char> terminals, nonterminals;

/* ---------- Utility ---------- */
bool isTerminal(char c) {
    return !isupper(c);
}

/* ---------- Compute FIRSTVT ---------- */
void computeFIRSTVT() {
    bool changed = true;

    while (changed) {
        changed = false;

        for (auto p : productions) {
            char A = p[0];
            char X = p[3];

            if (isTerminal(X)) {
                if (FIRSTVT[A].insert(X).second)
                    changed = true;
            }
            else {
                if (p.length() > 4 && isTerminal(p[4])) {
                    if (FIRSTVT[A].insert(p[4]).second)
                        changed = true;
                }

                for (char t : FIRSTVT[X]) {
                    if (FIRSTVT[A].insert(t).second)
                        changed = true;
                }
            }
        }
    }
}

/* ---------- Compute LASTVT ---------- */
void computeLASTVT() {
    bool changed = true;

    while (changed) {
        changed = false;

        for (auto p : productions) {
            char A = p[0];
            char X = p.back();

            if (isTerminal(X)) {
                if (LASTVT[A].insert(X).second)
                    changed = true;
            }
            else {
                if (p.length() > 4 && isTerminal(p[p.length() - 2])) {
                    if (LASTVT[A].insert(p[p.length() - 2]).second)
                        changed = true;
                }

                for (char t : LASTVT[X]) {
                    if (LASTVT[A].insert(t).second)
                        changed = true;
                }
            }
        }
    }
}

/* ---------- Build Precedence Table ---------- */
void buildPrecedenceTable() {
    for (auto p : productions) {
        for (int i = 3; i < p.length() - 1; i++) {
            char a = p[i];
            char b = p[i + 1];

            if (isTerminal(a) && isTerminal(b))
                precedence[a][b] = '=';

            if (isTerminal(a) && !isTerminal(b)) {
                for (char t : FIRSTVT[b])
                    precedence[a][t] = '<';
            }

            if (!isTerminal(a) && isTerminal(b)) {
                for (char t : LASTVT[a])
                    precedence[t][b] = '>';
            }
        }
    }

    for (char t : terminals) {
        precedence['$'][t] = '<';
        precedence[t]['$'] = '>';
    }

    precedence['$']['$'] = '=';
}

/* ---------- Get Top Terminal ---------- */
char topTerminal(stack<char> st) {
    while (!st.empty()) {
        if (isTerminal(st.top()) || st.top() == '$')
            return st.top();
        st.pop();
    }
    return '$';
}

/* ---------- Parsing ---------- */
void parse(string input) {
    stack<char> st;
    st.push('$');
    input += '$';

    int i = 0;

    cout << "\nSTACK\tINPUT\tACTION\n";

    while (true) {
        char a = topTerminal(st);
        char b = input[i];

        // Print stack
        stack<char> tmp = st;
        string s = "";
        while (!tmp.empty()) {
            s = tmp.top() + s;
            tmp.pop();
        }

        cout << s << "\t" << input.substr(i) << "\t";

        if (precedence[a][b] == '<' || precedence[a][b] == '=') {
            cout << "Shift\n";
            st.push(b);
            i++;
        }
        else if (precedence[a][b] == '>') {
            cout << "Reduce\n";
            char t;

            do {
                t = st.top();
                st.pop();
            } while (precedence[topTerminal(st)][t] != '<');

            st.push('E'); // reduced to non-terminal
        }
        else {
            cout << "Error\n";
            return;
        }

        if (a == '$' && b == '$') {
            cout << "\nExpression Accepted ✔\n";
            return;
        }
    }
}

/* ---------- MAIN ---------- */
int main() {
    int n;

    cout << "Enter number of productions: ";
    cin >> n;

    cout << "Enter productions (e.g. E->E+E):\n";

    for (int i = 0; i < n; i++) {
        string p;
        cin >> p;

        productions.push_back(p);
        nonterminals.insert(p[0]);

        for (int j = 3; j < p.length(); j++) {
            if (isTerminal(p[j]))
                terminals.insert(p[j]);
        }
    }

    computeFIRSTVT();
    computeLASTVT();
    buildPrecedenceTable();

    string input;
    cout << "\nEnter input string: ";
    cin >> input;

    parse(input);

    return 0;
}

In [ ]:
%%writefile slr(1).cpp

#include <bits/stdc++.h>
#include <sstream>
#include <queue>
#include <iomanip>
#include <algorithm>
using namespace std;

typedef pair<int, int> Item;  // (production_id, dot_position)
typedef set<Item> ItemSet;

struct SLRParser {
    string start_symbol;
    string augmented_start = "S'";

    vector<string> lhs;
    vector<vector<string>> rhs;

    set<string> nonterms;
    set<string> terms;

    map<string, set<string>> first;
    map<string, set<string>> follow;

    vector<ItemSet> states;
    map<ItemSet, int> state_map;

    vector<string> term_list;
    map<string, int> term_id;

    vector<string> nt_list;
    map<string, int> nt_id;

    vector<vector<string>> action_table;
    vector<vector<int>> goto_table;

    bool has_conflict = false;

    /* ---------- Closure ---------- */
    ItemSet closure(const ItemSet& items) {
        ItemSet closed = items;
        queue<Item> q;

        for (auto it : items)
            q.push(it);

        while (!q.empty()) {
            Item it = q.front();
            q.pop();

            int p = it.first;
            int d = it.second;

            if (d < (int)rhs[p].size()) {
                string sym = rhs[p][d];

                if (nonterms.count(sym)) {
                    for (int np = 0; np < (int)lhs.size(); np++) {
                        if (lhs[np] == sym) {
                            Item new_item = {np, 0};

                            if (closed.find(new_item) == closed.end()) {
                                closed.insert(new_item);
                                q.push(new_item);
                            }
                        }
                    }
                }
            }
        }
        return closed;
    }

    /* ---------- GOTO ---------- */
    ItemSet goto_func(const ItemSet& iset, const string& sym) {
        ItemSet moved;

        for (auto it : iset) {
            int p = it.first;
            int d = it.second;

            if (d < (int)rhs[p].size() && rhs[p][d] == sym) {
                moved.insert({p, d + 1});
            }
        }
        return closure(moved);
    }

    /* ---------- FIRST ---------- */
    void compute_first() {
        for (const auto& t : terms)
            first[t].insert(t);

        bool changed = true;

        while (changed) {
            changed = false;

            for (int p = 0; p < (int)lhs.size(); p++) {
                string A = lhs[p];
                const auto& beta = rhs[p];

                set<string> rhs_first;
                bool can_eps = true;

                for (const auto& X : beta) {
                    if (nonterms.count(X) == 0) {
                        rhs_first.insert(X);
                        can_eps = false;
                        break;
                    } else {
                        for (const auto& f : first[X]) {
                            if (f != "epsilon")
                                rhs_first.insert(f);
                        }

                        if (first[X].find("epsilon") == first[X].end()) {
                            can_eps = false;
                            break;
                        }
                    }
                }

                if (can_eps)
                    rhs_first.insert("epsilon");

                size_t old_sz = first[A].size();
                first[A].insert(rhs_first.begin(), rhs_first.end());

                if (first[A].size() > old_sz)
                    changed = true;
            }
        }
    }

    /* ---------- FOLLOW ---------- */
    void compute_follow() {
        follow[start_symbol].insert("$");

        bool changed = true;

        while (changed) {
            changed = false;

            for (int p = 0; p < (int)lhs.size(); p++) {
                string A = lhs[p];
                const auto& beta = rhs[p];

                for (size_t j = 0; j < beta.size(); j++) {
                    string B = beta[j];

                    if (nonterms.count(B) == 0)
                        continue;

                    set<string> trailer;
                    bool add_all = true;

                    for (size_t k = j + 1; k < beta.size(); k++) {
                        string Y = beta[k];

                        if (nonterms.count(Y) == 0) {
                            trailer.insert(Y);
                            add_all = false;
                            break;
                        } else {
                            for (const auto& f : first[Y]) {
                                if (f != "epsilon")
                                    trailer.insert(f);
                            }

                            if (first[Y].find("epsilon") == first[Y].end()) {
                                add_all = false;
                                break;
                            }
                        }
                    }

                    if (add_all) {
                        trailer.insert(follow[A].begin(), follow[A].end());
                    }

                    size_t old_sz = follow[B].size();
                    follow[B].insert(trailer.begin(), trailer.end());

                    if (follow[B].size() > old_sz)
                        changed = true;
                }
            }
        }
    }

    /* ---------- Build States ---------- */
    void build_states() {
        ItemSet initial = closure({{0, 0}});
        states.push_back(initial);
        state_map[initial] = 0;

        queue<int> q;
        q.push(0);

        while (!q.empty()) {
            int si = q.front();
            q.pop();

            set<string> moves;

            for (auto it : states[si]) {
                if (it.second < (int)rhs[it.first].size())
                    moves.insert(rhs[it.first][it.second]);
            }

            for (const auto& sym : moves) {
                ItemSet next = goto_func(states[si], sym);

                if (next.empty())
                    continue;

                if (state_map.count(next) == 0) {
                    int nid = states.size();
                    states.push_back(next);
                    state_map[next] = nid;
                    q.push(nid);
                }
            }
        }
    }

    /* ---------- Build Tables ---------- */
    void build_tables() {
        term_list.assign(terms.begin(), terms.end());
        term_list.push_back("$");

        for (int i = 0; i < (int)term_list.size(); i++)
            term_id[term_list[i]] = i;

        nt_list.assign(nonterms.begin(), nonterms.end());

        for (int i = 0; i < (int)nt_list.size(); i++)
            nt_id[nt_list[i]] = i;

        int num_states = states.size();

        action_table.assign(num_states, vector<string>(term_list.size(), ""));
        goto_table.assign(num_states, vector<int>(nt_list.size(), -1));

        // Shift & GOTO
        for (int i = 0; i < num_states; i++) {
            set<string> moves;

            for (auto it : states[i]) {
                if (it.second < (int)rhs[it.first].size()) {
                    moves.insert(rhs[it.first][it.second]);
                }
            }

            for (const auto& sym : moves) {
                ItemSet next = goto_func(states[i], sym);
                int j = state_map[next];

                if (nonterms.count(sym) == 0) {
                    int col = term_id.at(sym);

                    if (!action_table[i][col].empty())
                        has_conflict = true;

                    action_table[i][col] = "s" + to_string(j);
                } else {
                    int col = nt_id.at(sym);
                    goto_table[i][col] = j;
                }
            }
        }

        // Reduce & Accept
        for (int i = 0; i < num_states; i++) {
            for (auto it : states[i]) {
                if (it.second == (int)rhs[it.first].size()) {
                    int p = it.first;
                    string A = lhs[p];

                    if (p == 0) {
                        int col = term_id["$"];

                        if (!action_table[i][col].empty())
                            has_conflict = true;

                        action_table[i][col] = "acc";
                    } else {
                        for (const auto& a : follow[A]) {
                            int col = term_id.at(a);

                            if (!action_table[i][col].empty())
                                has_conflict = true;

                            action_table[i][col] = "r" + to_string(p);
                        }
                    }
                }
            }
        }
    }

    /* ---------- Production String ---------- */
    string get_production_str(int p) {
        ostringstream oss;
        oss << lhs[p] << " -> ";

        if (rhs[p].empty())
            oss << "epsilon";
        else {
            for (size_t k = 0; k < rhs[p].size(); k++) {
                oss << rhs[p][k];
                if (k + 1 < rhs[p].size())
                    oss << " ";
            }
        }
        return oss.str();
    }

    /* ---------- Parsing ---------- */
    void parse(const vector<string>& tokens) {
        vector<string> input = tokens;

        if (input.empty() || input.back() != "$")
            input.push_back("$");

        stack<int> state_stack;
        stack<string> symbol_stack;

        state_stack.push(0);

        int ip = 0;

        cout << "\nParsing steps:\n";
        cout << left << setw(40) << "Stack"
             << setw(40) << "Input"
             << "Action\n";

        cout << string(100, '-') << "\n";

        while (true) {
            int state = state_stack.top();

            string stack_str = "";
            stack<int> temp = state_stack;
            vector<int> vec;

            while (!temp.empty()) {
                vec.push_back(temp.top());
                temp.pop();
            }

            reverse(vec.begin(), vec.end());

            for (int x : vec)
                stack_str += to_string(x) + " ";

            string input_str = "";
            for (int j = ip; j < input.size(); j++)
                input_str += input[j] + " ";

            cout << setw(40) << stack_str
                 << setw(40) << input_str;

            string token = input[ip];
            int col = term_id[token];
            string act = action_table[state][col];

            if (act.empty()) {
                cout << "Error\n";
                break;
            }

            if (act[0] == 's') {
                cout << "Shift\n";
                int new_state = stoi(act.substr(1));
                state_stack.push(new_state);
                symbol_stack.push(token);
                ip++;
            }
            else if (act[0] == 'r') {
                int p = stoi(act.substr(1));
                cout << "Reduce " << get_production_str(p) << "\n";

                for (int k = 0; k < rhs[p].size(); k++) {
                    state_stack.pop();
                    symbol_stack.pop();
                }

                string A = lhs[p];
                symbol_stack.push(A);

                int new_state = goto_table[state_stack.top()][nt_id[A]];
                state_stack.push(new_state);
            }
            else if (act == "acc") {
                cout << "Accept\n";
                break;
            }
        }
    }
};

/* ---------- MAIN ---------- */
int main() {
    SLRParser parser;

    cout << "Enter start symbol: ";
    cin >> parser.start_symbol;
    cin.ignore();

    cout << "Enter productions (A -> alpha), empty line to stop:\n";

    string line;
    while (getline(cin, line)) {
        if (line.empty()) break;

        stringstream ss(line);
        string left, arrow;

        ss >> left >> arrow;

        vector<string> right;
        string sym;

        while (ss >> sym)
            right.push_back(sym);

        if (right.empty())
            right.push_back("epsilon");

        parser.lhs.push_back(left);
        parser.rhs.push_back(right);
    }

    parser.lhs.insert(parser.lhs.begin(), parser.augmented_start);
    parser.rhs.insert(parser.rhs.begin(), {parser.start_symbol});

    for (auto& l : parser.lhs)
        parser.nonterms.insert(l);

    for (auto& prod : parser.rhs) {
        for (auto& sym : prod) {
            if (!parser.nonterms.count(sym))
                parser.terms.insert(sym);
        }
    }

    parser.compute_first();
    parser.compute_follow();
    parser.build_states();
    parser.build_tables();

    if (parser.has_conflict) {
        cout << "Grammar is NOT SLR(1)\n";
        return 0;
    }

    cout << "\nGrammar is SLR(1)\n";

    cout << "\nEnter input string (tokens space separated): ";
    string input_line;
    getline(cin, input_line);

    stringstream ss(input_line);
    vector<string> tokens;
    string tok;

    while (ss >> tok)
        tokens.push_back(tok);

    parser.parse(tokens);

    return 0;
}

In [5]:
%%writefile syntaxtreeparser.cpp

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX_RULES 10
#define MAX_RHS 10

typedef struct {
    char lhs;
    char rhs[MAX_RHS];
} Rule;

typedef struct Node {
    char val[10];
    struct Node* children[MAX_RHS];
    int child_count;
} Node;

Rule grammar[MAX_RULES];
int num_rules = 0;

char input_str[100];
int cursor = 0;

/* ---------- Create Node ---------- */
Node* create_node(char* v) {
    Node* n = (Node*)malloc(sizeof(Node));

    strcpy(n->val, v);
    n->child_count = 0;

    for (int i = 0; i < MAX_RHS; i++)
        n->children[i] = NULL;

    return n;
}

/* ---------- Recursive Parsing ---------- */
Node* parse(char symbol) {

    // Epsilon
    if (symbol == '#') {
        return create_node("#");
    }

    // Terminal
    if (!(symbol >= 'A' && symbol <= 'Z')) {
        if (input_str[cursor] == symbol) {
            char temp[2] = { input_str[cursor++], '\0' };
            return create_node(temp);
        }
        return NULL;
    }

    // Non-terminal
    for (int i = 0; i < num_rules; i++) {
        if (grammar[i].lhs == symbol) {

            int saved_cursor = cursor;

            char temp[2] = { symbol, '\0' };
            Node* parent = create_node(temp);

            int possible = 1;

            for (int j = 0; grammar[i].rhs[j] != '\0'; j++) {
                Node* child = parse(grammar[i].rhs[j]);

                if (child != NULL) {
                    parent->children[parent->child_count++] = child;
                } else {
                    possible = 0;
                    break;
                }
            }

            if (possible)
                return parent;

            // Backtrack
            cursor = saved_cursor;
            free(parent);
        }
    }

    return NULL;
}

/* ---------- Print Tree ---------- */
void print_graphical(Node* root, char* prefix, int isLast) {
    if (!root) return;

    printf("%s%s%s\n", prefix,
           isLast ? "└── " : "├── ",
           root->val);

    char new_prefix[200];

    for (int i = 0; i < root->child_count; i++) {
        sprintf(new_prefix, "%s%s",
                prefix,
                isLast ? "    " : "│   ");

        print_graphical(
            root->children[i],
            new_prefix,
            i == root->child_count - 1
        );
    }
}

/* ---------- MAIN ---------- */
int main() {

    printf("Number of rules: ");
    scanf("%d", &num_rules);

    printf("Enter rules (e.g., E TN). Use '#' for epsilon:\n");

    for (int i = 0; i < num_rules; i++) {
        scanf(" %c %s", &grammar[i].lhs, grammar[i].rhs);
    }

    printf("Enter string: ");
    scanf("%s", input_str);

    Node* root = parse(grammar[0].lhs);

    if (root && (cursor == (int)strlen(input_str))) {
        printf("\nSYNTAX TREE:\n");
        print_graphical(root, "", 1);
    } else {
        printf("\nError: String could not be fully parsed.\n");
    }

    return 0;
}

SyntaxError: invalid syntax (3028326335.py, line 8)

In [5]:
%%writefile lalr.cpp

#include <bits/stdc++.h>
using namespace std;

/* ---------- Production ---------- */
struct Production {
    string lhs;
    vector<string> rhs;
};

vector<Production> grammar;
set<string> nonTerminals, terminals;
string startSymbol;

/* ---------- LR(1) Item ---------- */
struct Item {
    int prod;
    int dot;
    string lookahead;

    bool operator<(const Item &o) const {
        if (prod != o.prod) return prod < o.prod;
        if (dot != o.dot) return dot < o.dot;
        return lookahead < o.lookahead;
    }
};

using ItemSet = set<Item>;

/* ---------- FIRST ---------- */
map<string, set<string>> FIRST;

void computeFIRST() {
    for (auto t : terminals)
        FIRST[t].insert(t);

    bool changed = true;
    while (changed) {
        changed = false;

        for (auto &p : grammar) {
            string A = p.lhs;

            for (auto X : p.rhs) {
                for (auto f : FIRST[X]) {
                    if (f != "ε" && !FIRST[A].count(f)) {
                        FIRST[A].insert(f);
                        changed = true;
                    }
                }
                if (!FIRST[X].count("ε")) break;
            }
        }
    }
}

set<string> firstSeq(vector<string> &beta, string la) {
    set<string> res;

    for (auto &X : beta) {
        for (auto f : FIRST[X])
            if (f != "ε") res.insert(f);

        if (!FIRST[X].count("ε"))
            return res;
    }

    res.insert(la);
    return res;
}

/* ---------- Closure ---------- */
ItemSet closure(ItemSet I) {
    bool changed = true;

    while (changed) {
        changed = false;
        ItemSet temp = I;

        for (auto &it : temp) {
            auto &prod = grammar[it.prod];

            if (it.dot < prod.rhs.size()) {
                string B = prod.rhs[it.dot];

                if (nonTerminals.count(B)) {
                    vector<string> beta;

                    for (int k = it.dot + 1; k < prod.rhs.size(); k++)
                        beta.push_back(prod.rhs[k]);

                    auto look = firstSeq(beta, it.lookahead);

                    for (int i = 0; i < grammar.size(); i++) {
                        if (grammar[i].lhs == B) {
                            for (auto &l : look) {
                                Item newItem = {i, 0, l};

                                if (!I.count(newItem)) {
                                    I.insert(newItem);
                                    changed = true;
                                }
                            }
                        }
                    }
                }
            }
        }
    }
    return I;
}

/* ---------- GOTO ---------- */
ItemSet gotoSet(ItemSet I, string X) {
    ItemSet J;

    for (auto &it : I) {
        auto &prod = grammar[it.prod];

        if (it.dot < prod.rhs.size() && prod.rhs[it.dot] == X) {
            J.insert({it.prod, it.dot + 1, it.lookahead});
        }
    }
    return closure(J);
}

/* ---------- Canonical LR(1) ---------- */
vector<ItemSet> C;
map<pair<int,string>, int> trans;

void buildLR1() {
    ItemSet start;
    start.insert({0, 0, "$"});
    start = closure(start);

    C.push_back(start);

    for (int i = 0; i < C.size(); i++) {
        set<string> symbols;

        for (auto &it : C[i]) {
            auto &p = grammar[it.prod];
            if (it.dot < p.rhs.size())
                symbols.insert(p.rhs[it.dot]);
        }

        for (auto X : symbols) {
            ItemSet J = gotoSet(C[i], X);
            if (J.empty()) continue;

            int id = -1;
            for (int k = 0; k < C.size(); k++)
                if (C[k] == J) id = k;

            if (id == -1) {
                id = C.size();
                C.push_back(J);
            }

            trans[{i, X}] = id;
        }
    }
}

/* ---------- Merge to LALR ---------- */
map<set<pair<int,int>>, int> coreMap;
vector<ItemSet> LALR;

void buildLALR() {
    for (int i = 0; i < C.size(); i++) {
        set<pair<int,int>> core;

        for (auto &it : C[i])
            core.insert({it.prod, it.dot});

        if (!coreMap.count(core)) {
            coreMap[core] = LALR.size();
            LALR.push_back(ItemSet());
        }

        int id = coreMap[core];

        for (auto &it : C[i])
            LALR[id].insert(it);
    }
}

/* ---------- Tables ---------- */
map<pair<int,string>, string> ACTION;
map<pair<int,string>, int> GOTO;

void buildTable() {
    for (int i = 0; i < LALR.size(); i++) {
        for (auto &it : LALR[i]) {
            auto &p = grammar[it.prod];

            if (it.dot < p.rhs.size()) {
                string a = p.rhs[it.dot];

                if (terminals.count(a)) {
                    int j = trans[{i, a}];
                    ACTION[{i, a}] = "s" + to_string(j);
                }
            }
            else {
                if (p.lhs == "S'") {
                    ACTION[{i, "$"}] = "acc";
                }
                else {
                    ACTION[{i, it.lookahead}] =
                        "r" + to_string(it.prod);
                }
            }
        }

        for (auto nt : nonTerminals) {
            if (trans.count({i, nt}))
                GOTO[{i, nt}] = trans[{i, nt}];
        }
    }
}

/* ---------- Print ---------- */
void printStates() {
    for (int i = 0; i < LALR.size(); i++) {
        cout << "I" << i << ":\n";
        for (auto &it : LALR[i]) {
            auto &p = grammar[it.prod];

            cout << p.lhs << " -> ";

            for (int j = 0; j <= p.rhs.size(); j++) {
                if (j == it.dot) cout << ". ";
                if (j < p.rhs.size()) cout << p.rhs[j] << " ";
            }

            cout << ", " << it.lookahead << "\n";
        }
        cout << "\n";
    }
}

/* ---------- Parsing ---------- */
void parse(vector<string> input) {
    stack<int> st;
    st.push(0);

    int i = 0;

    while (true) {
        int s = st.top();
        string a = input[i];

        string act = ACTION[{s, a}];

        if (act[0] == 's') {
            st.push(stoi(act.substr(1)));
            i++;
        }
        else if (act[0] == 'r') {
            int p = stoi(act.substr(1));

            for (int k = 0; k < grammar[p].rhs.size(); k++)
                st.pop();

            int t = st.top();
            st.push(GOTO[{t, grammar[p].lhs}]);
        }
        else if (act == "acc") {
            cout << "ACCEPTED\n";
            return;
        }
        else {
            cout << "ERROR\n";
            return;
        }
    }
}

/* ---------- MAIN ---------- */
int main() {
    int n;
    cin >> n;
    cin.ignore();

    for (int i = 0; i < n; i++) {
        string line;
        getline(cin, line);

        stringstream ss(line);
        string lhs, arrow;
        ss >> lhs >> arrow;

        vector<string> rhs;
        string x;
        while (ss >> x) rhs.push_back(x);

        grammar.push_back({lhs, rhs});
        nonTerminals.insert(lhs);

        if (i == 0) startSymbol = lhs;
    }

    grammar.insert(grammar.begin(), {"S'", {startSymbol}});
    nonTerminals.insert("S'");

    for (auto &p : grammar)
        for (auto &x : p.rhs)
            if (!nonTerminals.count(x))
                terminals.insert(x);

    terminals.insert("$");

    computeFIRST();
    buildLR1();
    buildLALR();
    buildTable();

    printStates();

    vector<string> input = {"id","+","id","$"};
    parse(input);

    return 0;
}

In [ ]:
%%writefile tacgen.cpp

#include <iostream>
#include <stack>
#include <string>
#include <vector>
#include <cctype>

using namespace std;

/* ---------- Operator Precedence ---------- */
int precedence(char op) {
    if (op == '+' || op == '-') return 1;
    if (op == '*' || op == '/') return 2;
    return 0;
}

/* ---------- Infix to Postfix ---------- */
string infixToPostfix(string expression) {
    stack<char> s;
    string postfix = "";

    for (char c : expression) {

        // Operand
        if (isalnum(c)) {
            postfix += c;
        }

        // Opening bracket
        else if (c == '(') {
            s.push(c);
        }

        // Closing bracket
        else if (c == ')') {
            while (!s.empty() && s.top() != '(') {
                postfix += s.top();
                s.pop();
            }
            if (!s.empty()) s.pop(); // remove '('
        }

        // Operator
        else {
            while (!s.empty() && precedence(s.top()) >= precedence(c)) {
                postfix += s.top();
                s.pop();
            }
            s.push(c);
        }
    }

    // Pop remaining operators
    while (!s.empty()) {
        postfix += s.top();
        s.pop();
    }

    return postfix;
}

/* ---------- Generate Three Address Code ---------- */
void generateTAC(string postfix) {
    stack<string> s;
    int tempCount = 1;

    for (char c : postfix) {

        // Operand
        if (isalnum(c)) {
            string operand(1, c);
            s.push(operand);
        }

        // Operator
        else {
            string op2 = s.top(); s.pop();
            string op1 = s.top(); s.pop();

            string res = "t" + to_string(tempCount++);

            cout << res << " = " << op1 << " " << c << " " << op2 << endl;

            s.push(res);
        }
    }
}

/* ---------- MAIN ---------- */
int main() {
    string expression;

    cout << "Enter an arithmetic expression (e.g., a+b*c): ";
    cin >> expression;

    string postfix = infixToPostfix(expression);

    cout << "\nThree-Address Code Generation:\n";
    generateTAC(postfix);

    return 0;
}